In [1]:
!pip install wandb -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 84.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import wandb

In [3]:
SEED = 42
SR = 22050
DURATION = 30        # seconds
N_MELS = 128
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-3
MAX_FRAMES = 1300 

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = secret_value_0
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sharibahmad (sharibahmad-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
print("W&B user:", wandb.api.viewer())

W&B user: {'id': 'VXNlcjozOTE5MDI2', 'entity': 'sharibahmad-indian-institute-of-technology-madras', 'username': 'sharibahmad', 'flags': '{"name":"default","rate_limit":"400/s","system_metrics":"2/m","sweeps_enabled":false,"teams_enabled":false,"private_projects":true,"gpu_enabled":null,"hub_settings":{"repo":"lukas/ml-class","disk":"10Gi","expiration":259200,"redis_enabled":false,"docker_enabled":false,"image":null},"restricted":false,"proxy_settings":{"openai":null},"noContact":false}', 'teams': {'edges': [{'node': {'name': 'sharibahmad'}}, {'node': {'name': 'sharibahmad-indian-institute-of-technology-madras'}}]}}


In [6]:
wandb.init(
    project="24f2001786-t12026",
    name="cnn-baseline",
    config={
        "model": "SimpleCNN",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "n_mels": N_MELS
    }
)

In [7]:
GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

In [8]:
DATA_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

samples = []

for genre in GENRES:
    genre_path = os.path.join(DATA_DIR, genre)
    for song in os.listdir(genre_path):
        audio_path = os.path.join(genre_path, song, "other.wav")
        if os.path.exists(audio_path):
            samples.append((audio_path, GENRE_TO_IDX[genre]))

len(samples)

1000

In [9]:
train_samples, val_samples = train_test_split(
    samples, test_size=0.2, random_state=SEED, stratify=[s[1] for s in samples]
)

In [10]:
class AudioDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        y, _ = librosa.load(path, sr=SR, duration=DURATION)

        mel = librosa.feature.melspectrogram(
            y=y,
            sr=SR,
            n_mels=N_MELS
        )
        mel = librosa.power_to_db(mel)

        # 🔑 FIX: pad or trim time dimension
        if mel.shape[1] < MAX_FRAMES:
            pad_width = MAX_FRAMES - mel.shape[1]
            mel = np.pad(mel, ((0, 0), (0, pad_width)))
        else:
            mel = mel[:, :MAX_FRAMES]

        mel = torch.tensor(mel).unsqueeze(0)  # (1, n_mels, time)
        return mel, label

In [11]:
train_ds = AudioDataset(train_samples)
val_ds = AudioDataset(val_samples)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [13]:
model = SimpleCNN(len(GENRES)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [14]:
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            out = model(x)
            preds.extend(out.argmax(1).cpu().numpy())
            targets.extend(y.numpy())

    f1 = f1_score(targets, preds, average="macro")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss / len(train_loader),
        "val_macro_f1": f1
    })

    print(f"Epoch {epoch} | F1: {f1:.4f}")

Epoch 0 | F1: 0.1115
Epoch 1 | F1: 0.1770
Epoch 2 | F1: 0.1223
Epoch 3 | F1: 0.1865
Epoch 4 | F1: 0.1556


In [19]:
TEST_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

In [20]:
def predict_file(path):
    y, _ = librosa.load(path, sr=SR, duration=DURATION)
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS)
    mel = librosa.power_to_db(mel)

    x = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    return IDX_TO_GENRE[out.argmax(1).item()]

In [21]:
preds = []

for _, row in test_df.iterrows():
    audio_path = os.path.join(TEST_DIR, row["filename"])
    preds.append(predict_file(audio_path))

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": preds
})

submission.to_csv("submission_cnn_baseline.csv", index=False)
submission.head()

,id,genre
0,1,metal
1,2,blues
2,3,metal
3,4,metal
4,5,metal


In [22]:
artifact = wandb.Artifact("cnn-baseline-submission", type="submission")
artifact.add_file("submission_cnn_baseline.csv")
wandb.log_artifact(artifact)

wandb.finish()

epoch,▁▃▅▆█
train_loss,█▂▂▁▁
val_macro_f1,▁▇▂█▅
epoch,4
train_loss,2.1335
val_macro_f1,0.15565
